In [9]:
from pathlib import Path
import matplotlib.pyplot as plt
import tensorflow as tf

# Config
IMAGE_SIZE = 512
BATCH_SIZE = 8
TEST_BATCH_SIZE = 1

TRAIN_IMAGE_FOLDER = Path("Segmentation/Train/Image")
TRAIN_MASK_FOLDER  = Path("Segmentation/Train/Masks")

VAL_IMAGE_FOLDER   = Path("Segmentation/Val/Image")
VAL_MASK_FOLDER    = Path("Segmentation/Val/Masks")

TEST_IMAGE_FOLDER  = Path("Segmentation/Test/Image")
TEST_MASK_FOLDER   = Path("Segmentation/Test/Masks")

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_EXTS  = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


# File Pairing
def collect_files(folder, exts):
    return [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts]


def build_pairs(image_folder, mask_folder):
    image_files = collect_files(image_folder, IMAGE_EXTS)
    mask_files = collect_files(mask_folder, MASK_EXTS)

    image_map = {p.stem: p for p in image_files}
    mask_map  = {p.stem: p for p in mask_files}

    common_stems = sorted(set(image_map.keys()) & set(mask_map.keys()))

    image_paths = [str(image_map[s]) for s in common_stems]
    mask_paths  = [str(mask_map[s]) for s in common_stems]

    print(f"Matched pairs from {image_folder} and {mask_folder}: {len(common_stems)}")
    return image_paths, mask_paths


train_image_paths, train_mask_paths = build_pairs(TRAIN_IMAGE_FOLDER, TRAIN_MASK_FOLDER)
val_image_paths, val_mask_paths     = build_pairs(VAL_IMAGE_FOLDER, VAL_MASK_FOLDER)
test_image_paths, test_mask_paths   = build_pairs(TEST_IMAGE_FOLDER, TEST_MASK_FOLDER)

# Reading
def read_image(path, mask=False):
    x = tf.io.read_file(path)
    x = tf.image.decode_image(x, channels=1 if mask else 3, expand_animations=False)
    x = tf.cast(x, tf.float32) / 255.0

    if mask:
        x = tf.where(x >= 0.5, 1.0, 0.0)
        x = tf.image.resize(
            x,
            [IMAGE_SIZE, IMAGE_SIZE],
            method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
        )
        x.set_shape([IMAGE_SIZE, IMAGE_SIZE, 1])
    else:
        x = tf.image.resize(
            x,
            [IMAGE_SIZE, IMAGE_SIZE],
            method=tf.image.ResizeMethod.BILINEAR
        )
        x.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3])

    return x


def load_train_val_data(image_path, mask_path):
    image = read_image(image_path, mask=False)
    mask = read_image(mask_path, mask=True)
    return image, mask


def load_test_data(image_path, mask_path):
    image = read_image(image_path, mask=False)
    mask = read_image(mask_path, mask=True)
    return image, mask


# Train Augmentation Only
def augment_data(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    if tf.random.uniform(()) > 0.5:
        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        image = tf.image.rot90(image, k=k)
        mask = tf.image.rot90(mask, k=k)

    image = tf.clip_by_value(image, 0.0, 1.0)
    mask = tf.where(mask >= 0.5, 1.0, 0.0)
    return image, mask

# Dataset Builders
def build_dataset(image_list, mask_list, augment=False, batch_size=8):
    dataset = tf.data.Dataset.from_tensor_slices((image_list, mask_list))
    dataset = dataset.map(load_train_val_data, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        dataset = dataset.map(augment_data, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


def build_test_dataset(image_list, mask_list, batch_size=1):
    dataset = tf.data.Dataset.from_tensor_slices((image_list, mask_list))
    dataset = dataset.map(load_test_data, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(train_image_paths, train_mask_paths, augment=True, batch_size=BATCH_SIZE)
val_dataset = build_dataset(val_image_paths, val_mask_paths, augment=False, batch_size=BATCH_SIZE)
test_dataset = build_test_dataset(test_image_paths, test_mask_paths, batch_size=TEST_BATCH_SIZE)

print("Train Dataset:", train_dataset)
print("Val Dataset:", val_dataset)
print("Test Dataset:", test_dataset)

# plot only masks (as PMI images are sensitive, only masks are plotted)
# uncomment for plotting

# def plot_train_masks(dataset, num_masks=3):
#     for images, masks in dataset.take(1):
#         num_masks = min(num_masks, masks.shape[0])

#         plt.figure(figsize=(10, 3.5))
#         for i in range(num_masks):
#             mask = masks[i].numpy().squeeze()

#             plt.subplot(1, num_masks, i + 1)
#             plt.imshow(mask, cmap="gray")
#             plt.title(f"Mask {i+1}")
#             plt.axis("off")

#         plt.tight_layout()
#         plt.show()

# plot_train_masks(train_dataset, num_masks=3)

Matched pairs from Segmentation/Train/Image and Segmentation/Train/Masks: 449
Matched pairs from Segmentation/Val/Image and Segmentation/Val/Masks: 145
Matched pairs from Segmentation/Test/Image and Segmentation/Test/Masks: 147
Train Dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 512, 512, 1), dtype=tf.float32, name=None))>
Val Dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 512, 512, 1), dtype=tf.float32, name=None))>
Test Dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 512, 512, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 512, 512, 1), dtype=tf.float32, name=None))>
